In [ ]:
%load_ext autoreload
%autoreload 2

In [46]:
# Let's start with necessary imports
import os
import numpy as np
from shutil import copyfile
from tqdm.notebook import tqdm
from matplotlib import pyplot as plt

## Define Settings

In [64]:
data_dir = "../../data/images/"
meta_data_path = "../../data/butterfly_anomaly_train.csv"
plot_dir = "../../plots/"

In [65]:
if not os.path.exists(plot_dir):
    os.makedirs(plot_dir)

## Define Data Loader

In [19]:
from hdr_hybrid_butterflies.data_handler import DataHandler

data_handler = DataHandler(
    meta_data_path=meta_data_path,
    data_dir=data_dir,
)

In [ ]:
data_handler.df_meta.iloc[10]["hybrid_stat"] == "nonhybrid"

In [ ]:
data_handler.df_meta.iloc[10]

In [ ]:
data_handler.load_data(1804)[0]

### Copy Files of Subspecies to sub-directories

In [ ]:
np.unique(data_handler.df_meta["subspecies"], return_counts=True)

In [38]:
if False:
    for idx, row in tqdm(
        data_handler.df_meta.iterrows(), total=data_handler.n_samples
    ):
        if row["hybrid_stat"] == "non-hybrid":
            input_path = os.path.join(
                data_dir, row["hybrid_stat"], row["filename"]
            )
            output_path = os.path.join(
                data_dir,
                row["hybrid_stat"],
                f"{int(row['subspecies']):02d}",
                row["filename"],
            )
            output_dir = os.path.dirname(output_path)
            if not os.path.exists(output_dir):
                os.makedirs(output_dir)
            copyfile(input_path, output_path)

## Create overview of all Subspecies

In [ ]:
n_subspecies = len(np.unique(data_handler.df_meta["subspecies"]))
n_subspecies

In [ ]:
data_handler.df_meta[
    data_handler.df_meta["subspecies"].isnull()
].parent_subspecies_2.unique()

In [ ]:
for seed in range(10):
    rng = np.random.default_rng(seed)

    sub_species = np.unique(data_handler.df_meta["subspecies"])

    fig, axes = plt.subplots(3, 5, figsize=(30, 15))
    axes_flat = axes.flatten()

    for idx, sub in enumerate(sub_species):
        if np.isnan(sub):
            indices = data_handler.df_meta[
                data_handler.df_meta["subspecies"].isnull()
            ].index
        else:
            indices = data_handler.df_meta[
                data_handler.df_meta["subspecies"] == sub
            ].index

        chosen_idx = rng.choice(indices)

        img, row = data_handler.load_data(chosen_idx)
        axes_flat[idx].imshow(img)
        axes_flat[idx].set_title(
            f"Subspecies: {row['subspecies']} | Idx: {chosen_idx} | Occurrences: {len(indices)}"
        )
        axes_flat[idx].axis("off")

    plt.tight_layout()
    fig.savefig(os.path.join(plot_dir, f"subspecies_examples_{seed:04d}.png"))